# 🌫️ Delhi Air Pollution: The Data Behind the "Blame Game"

**Project by Asrar**

### Research question
Do pollution levels in Punjab and Haryana show a time-lagged relationship with Delhi's PM2.5, and can those signals help predict Delhi pollution and severe days?

**Dataset:** [Time Series Air Quality Data of India (2010–2023)](https://www.kaggle.com/datasets/abhisheksjha/time-series-air-quality-data-of-india-2010-2023)

### Analysis flow
Raw station data → station-aware filtering → daily PM2.5 → lag/rolling features → dual-era correlation → COVID/stubble analysis → regression → ablation → severe-day classification.

> **Important:** Correlation and lagged prediction show association/temporal predictive value; they do not by themselves prove causality.


In [ ]:
# 1. Setup
import os
import glob
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import xgboost as xgb

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVC
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.utils import class_weight

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Change only this path if your dataset is stored elsewhere.
DATA_PATH = r"C:\Users\asrar\.cache\kagglehub\datasets\abhisheksjha\time-series-air-quality-data-of-india-2010-2023\versions\aqidataset"

print(f"Data path: {DATA_PATH}")
print(f"Files found: {len(os.listdir(DATA_PATH))}")


In [ ]:
# 2. Station metadata
stations_df = pd.read_csv(os.path.join(DATA_PATH, "stations_info.csv"))
stations_df["start_year"] = pd.to_numeric(stations_df["start_year"], errors="coerce")

required = {"file_name", "start_year", "state", "city"}
missing = required - set(stations_df.columns)
if missing:
    raise ValueError(f"Missing required metadata columns: {sorted(missing)}")

state_counts = {
    "Delhi": stations_df["file_name"].str.startswith("DL").sum(),
    "Haryana": stations_df["file_name"].str.startswith("HR").sum(),
    "Punjab": stations_df["file_name"].str.startswith("PB").sum()
}

print(f"Stations: {len(stations_df)}")
print(f"Start years: {stations_df.start_year.min():.0f}–{stations_df.start_year.max():.0f}")
print("State stations:", state_counts)

stations_df.head()


In [ ]:
# 3. Load station data, keeping only valid PM2.5 observations
# The original analysis loads at most 20 files per state; that limit is kept unchanged.
MAX_STATIONS = 20

def load_state_data(prefix):
    files = [
        f for f in os.listdir(DATA_PATH)
        if f.startswith(prefix) and f.endswith(".csv")
    ][:MAX_STATIONS]

    rows = []

    for file in files:
        station = file[:-4]
        try:
            raw = pd.read_csv(os.path.join(DATA_PATH, file))
            pm_col = next((c for c in raw.columns if "pm2.5" in c.lower()), None)
            date_col = next((c for c in raw.columns if "date" in c.lower()), None)

            if not pm_col or not date_col:
                continue

            info = stations_df[stations_df["file_name"] == station]
            start_year = info["start_year"].iloc[0] if not info.empty else np.nan
            city = info["city"].iloc[0] if not info.empty else None

            df = raw[[date_col, pm_col]].copy()
            df.columns = ["date", "PM2.5"]
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df["PM2.5"] = pd.to_numeric(df["PM2.5"], errors="coerce")

            if pd.notna(start_year):
                df = df[df["date"].dt.year >= start_year]

            df = df[(df["PM2.5"] > 0) & (df["PM2.5"] <= 1000)]
            df["station"] = station
            df["start_year"] = start_year
            df["city"] = city
            rows.append(df)

        except Exception:
            continue

    if not rows:
        return None

    return pd.concat(rows, ignore_index=True)

delhi = load_state_data("DL")
haryana = load_state_data("HR")
punjab = load_state_data("PB")

for name, df in [("Delhi", delhi), ("Haryana", haryana), ("Punjab", punjab)]:
    print(
        f"{name}: {len(df):,} hourly rows | "
        f"{df.station.nunique()} stations | "
        f"{df.date.min().date()} → {df.date.max().date()}"
    )


In [ ]:
# 4. Daily averages + common analysis timeline
def daily_average(df, state):
    out = (
        df.set_index("date")["PM2.5"]
        .resample("D")
        .mean()
        .rename(f"{state}_PM2.5")
        .reset_index()
    )
    return out

daily = daily_average(delhi, "Delhi")
daily = daily.merge(daily_average(haryana, "Haryana"), on="date", how="left")
daily = daily.merge(daily_average(punjab, "Punjab"), on="date", how="left")

daily["year"] = daily.date.dt.year
daily["month"] = daily.date.dt.month
daily["is_stubble_season"] = (
    ((daily.month == 10) & (daily.date.dt.day >= 15)) | (daily.month == 11)
)
daily["era"] = np.select(
    [daily.year.between(2015, 2019), daily.year.between(2020, 2023)],
    ["Era_1_Pre_2019", "Era_2_Post_2020"],
    default="Other"
)

daily["covid_phase"] = np.select(
    [
        daily.date < "2020-03-01",
        daily.date.between("2020-03-01", "2020-06-30")
    ],
    ["Pre-COVID", "Lockdown"],
    default="Post-COVID"
)

print(f"Daily merged data: {daily.shape}")
print(f"Date range: {daily.date.min().date()} → {daily.date.max().date()}")
daily.head()


In [ ]:
# 5. Missing values + lag/rolling/time features
df_complete = daily[daily.date >= "2017-03-03"].copy()

for col in ["Haryana_PM2.5", "Punjab_PM2.5"]:
    df_complete[col] = df_complete[col].ffill().bfill()

df_complete["Delhi_PM2.5"] = df_complete["Delhi_PM2.5"].interpolate()

# Primary lags: Punjab = 2 days, Haryana = 1 day.
df_complete["Punjab_PM2.5_Lag_2Days"] = df_complete["Punjab_PM2.5"].shift(2)
df_complete["Haryana_PM2.5_Lag_1Day"] = df_complete["Haryana_PM2.5"].shift(1)

# Robustness lags.
df_complete["Punjab_PM2.5_Lag_1Day"] = df_complete["Punjab_PM2.5"].shift(1)
df_complete["Haryana_PM2.5_Lag_2Days"] = df_complete["Haryana_PM2.5"].shift(2)

# 3-day smoothing.
for state in ["Delhi", "Punjab", "Haryana"]:
    df_complete[f"{state}_PM2.5_3DayAvg"] = (
        df_complete[f"{state}_PM2.5"].rolling(3, min_periods=1).mean()
    )

df_complete["day_of_week"] = df_complete.date.dt.dayofweek
df_complete["is_weekend"] = (df_complete.day_of_week >= 5).astype(int)

df_complete = df_complete.dropna().reset_index(drop=True)

print(f"Final analysis data: {df_complete.shape}")
print(f"Dates: {df_complete.date.min().date()} → {df_complete.date.max().date()}")
print(f"Stubble-season days: {df_complete.is_stubble_season.sum()}")
print(df_complete[
    ["date", "Delhi_PM2.5", "Punjab_PM2.5_Lag_2Days",
     "Haryana_PM2.5_Lag_1Day", "is_stubble_season", "era", "covid_phase"]
].head())


In [ ]:
# 6. Dual-era correlation analysis
CORR_FEATURES = [
    "Punjab_PM2.5_Lag_2Days", "Punjab_PM2.5_Lag_1Day",
    "Haryana_PM2.5_Lag_1Day", "Haryana_PM2.5_Lag_2Days",
    "Punjab_PM2.5", "Haryana_PM2.5"
]

def correlations(df, era):
    rows = []
    for feature in CORR_FEATURES:
        r, p = stats.pearsonr(df["Delhi_PM2.5"], df[feature])
        rows.append({
            "Era": era,
            "Feature": feature,
            "r": r,
            "R2": r**2,
            "p": p
        })
    return pd.DataFrame(rows)

era1 = df_complete[df_complete.era == "Era_1_Pre_2019"]
era2 = df_complete[df_complete.era == "Era_2_Post_2020"]

corr_df = pd.concat([
    correlations(era1, "Era 1 (2015–2019)"),
    correlations(era2, "Era 2 (2020–2023)")
])

corr_df


In [ ]:
# 7. Stubble + COVID analysis
stubble_rows = []
for era_name, data in [
    ("Era 1", era1),
    ("Era 2", era2)
]:
    for label, subset in [
        ("Stubble", data[data.is_stubble_season]),
        ("Non-stubble", data[~data.is_stubble_season])
    ]:
        r, p = stats.pearsonr(subset["Delhi_PM2.5"], subset["Punjab_PM2.5_Lag_2Days"])
        stubble_rows.append({
            "Era": era_name, "Period": label, "r": r, "p": p,
            "Delhi_PM2.5_mean": subset["Delhi_PM2.5"].mean()
        })

stubble_df = pd.DataFrame(stubble_rows)

covid_df = (
    df_complete.groupby("covid_phase")[["Delhi_PM2.5", "Punjab_PM2.5", "Haryana_PM2.5"]]
    .mean()
    .reindex(["Pre-COVID", "Lockdown", "Post-COVID"])
)

covid_change = (
    (covid_df["Delhi_PM2.5"] - covid_df.loc["Pre-COVID", "Delhi_PM2.5"])
    / covid_df.loc["Pre-COVID", "Delhi_PM2.5"] * 100
)

print("Punjab 2-day lag correlation:")
print(corr_df[corr_df.Feature == "Punjab_PM2.5_Lag_2Days"][["Era", "r", "R2", "p"]].to_string(index=False))

print("\nHaryana 1-day lag correlation:")
print(corr_df[corr_df.Feature == "Haryana_PM2.5_Lag_1Day"][["Era", "r", "R2", "p"]].to_string(index=False))

print("\nStubble analysis:")
display(stubble_df)

print("\nCOVID averages:")
display(covid_df.round(1))
print("\nDelhi change vs pre-COVID (%):")
display(covid_change.round(1))


In [ ]:
# 8. Four analysis visuals
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# A. Dual-era lag correlations
key = corr_df[corr_df.Feature.isin([
    "Punjab_PM2.5_Lag_2Days", "Punjab_PM2.5_Lag_1Day",
    "Haryana_PM2.5_Lag_1Day", "Haryana_PM2.5_Lag_2Days"
])].copy()
plot_key = key.pivot(index="Feature", columns="Era", values="r")
plot_key.plot(kind="bar", ax=axes[0, 0])
axes[0, 0].set_title("Dual-Era Correlation with Delhi")
axes[0, 0].set_ylabel("Pearson r")
axes[0, 0].tick_params(axis="x", rotation=30)

# B. COVID averages
covid_df.plot(kind="bar", ax=axes[0, 1])
axes[0, 1].set_title("Average PM2.5 by COVID Phase")
axes[0, 1].set_ylabel("PM2.5 (µg/m³)")
axes[0, 1].tick_params(axis="x", rotation=0)

# C. Stubble vs non-stubble
stubble_plot = stubble_df.pivot(index="Period", columns="Era", values="r")
stubble_plot.plot(kind="bar", ax=axes[1, 0])
axes[1, 0].set_title("Punjab 2-Day Lag Correlation")
axes[1, 0].set_ylabel("Pearson r")
axes[1, 0].tick_params(axis="x", rotation=0)

# D. 2022 lag alignment
s22 = df_complete[(df_complete.year == 2022) & df_complete.month.isin([10, 11])]
axes[1, 1].plot(s22.date, s22["Delhi_PM2.5"], label="Delhi today")
axes[1, 1].plot(
    s22.date - pd.Timedelta(days=2),
    s22["Punjab_PM2.5_Lag_2Days"],
    label="Punjab (2-day lag)"
)
axes[1, 1].plot(
    s22.date - pd.Timedelta(days=1),
    s22["Haryana_PM2.5_Lag_1Day"],
    label="Haryana (1-day lag)"
)
axes[1, 1].set_title("Lag-Aligned Time Series — Oct/Nov 2022")
axes[1, 1].set_ylabel("PM2.5 (µg/m³)")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("dual_era_analysis.png", dpi=250, bbox_inches="tight")
plt.show()


In [ ]:
# 9. Regression: 70/30 time-based split
feature_cols = [
    "Punjab_PM2.5_Lag_2Days",
    "Haryana_PM2.5_Lag_1Day",
    "Punjab_PM2.5_Lag_1Day",
    "Haryana_PM2.5_Lag_2Days",
    "is_stubble_season",
    "month",
    "is_weekend",
    "Punjab_PM2.5_3DayAvg",
    "day_of_week"
]

X = df_complete[feature_cols]
y = df_complete["Delhi_PM2.5"]

split_idx = int(0.70 * len(X))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def evaluate_regression(model, name, Xtr, Xte):
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    return {
        "Model": name,
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    }, model, pred

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1,
        max_depth=10, min_samples_split=5
    )
}

results = []
trained_models = {}

for name, model in models.items():
    result, trained, _ = evaluate_regression(
        model, name, X_train_scaled, X_test_scaled
    )
    results.append(result)
    trained_models[name] = trained

# XGBoost: use GPU if available, otherwise CPU.
try:
    xgb_model = xgb.XGBRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=6,
        random_state=42, tree_method="hist", device="cuda",
        n_jobs=-1, subsample=0.8, colsample_bytree=0.8
    )
    xgb_name = "XGBoost"
    result, trained, _ = evaluate_regression(
        xgb_model, xgb_name, X_train_scaled, X_test_scaled
    )
except Exception:
    xgb_model = xgb.XGBRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=6,
        random_state=42, n_jobs=-1, subsample=0.8, colsample_bytree=0.8
    )
    xgb_name = "XGBoost"
    result, trained, _ = evaluate_regression(
        xgb_model, xgb_name, X_train_scaled, X_test_scaled
    )

results.append(result)
trained_models[xgb_name] = trained

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Train dates: {df_complete.date.iloc[0].date()} → {df_complete.date.iloc[split_idx-1].date()}")
print(f"Test dates: {df_complete.date.iloc[split_idx].date()} → {df_complete.date.iloc[-1].date()}")
display(results_df.round(3))


In [ ]:
# 10. Punjab + Haryana ablation studies
def ablation(group):
    keep = [i for i, col in enumerate(feature_cols) if group not in col]
    Xtr, Xte = X_train_scaled[:, keep], X_test_scaled[:, keep]

    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(
            n_estimators=100, random_state=42, n_jobs=-1,
            max_depth=10, min_samples_split=5
        )
    }

    # Same GPU-first XGBoost setup as the full model, with CPU fallback.
    models["XGBoost"] = xgb.XGBRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=6,
        random_state=42, tree_method="hist", device="cuda",
        n_jobs=-1, subsample=0.8, colsample_bytree=0.8
    )

    rows = []
    for name, model in models.items():
        try:
            model.fit(Xtr, y_train)
        except Exception:
            model = xgb.XGBRegressor(
                n_estimators=100, learning_rate=0.1, max_depth=6,
                random_state=42, n_jobs=-1
            )
            model.fit(Xtr, y_train)

        pred = model.predict(Xte)
        rows.append({
            "Model": name,
            "R2_without": r2_score(y_test, pred)
        })
    return pd.DataFrame(rows)

no_punjab = ablation("Punjab").rename(columns={"R2_without": "No_Punjab_R2"})
no_haryana = ablation("Haryana").rename(columns={"R2_without": "No_Haryana_R2"})

ablation_df = results_df[["Model", "R2"]].merge(no_punjab, on="Model")
ablation_df = ablation_df.merge(no_haryana, on="Model")
ablation_df["Punjab_relative_drop_%"] = (
    (ablation_df.R2 - ablation_df.No_Punjab_R2) / ablation_df.R2 * 100
)
ablation_df["Haryana_relative_drop_%"] = (
    (ablation_df.R2 - ablation_df.No_Haryana_R2) / ablation_df.R2 * 100
)

display(ablation_df.round(3))

print("Average Punjab relative reduction:",
      f"{ablation_df['Punjab_relative_drop_%'].mean():.1f}%")
print("Average Haryana relative reduction:",
      f"{ablation_df['Haryana_relative_drop_%'].mean():.1f}%")


In [ ]:
# 11. Regression visuals + feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full vs no-Punjab R²
ablation_df.set_index("Model")[["R2", "No_Punjab_R2"]].plot(kind="bar", ax=axes[0])
axes[0].set_title("Regression: Full vs No Punjab")
axes[0].set_ylabel("Test R²")
axes[0].tick_params(axis="x", rotation=20)

# Random Forest feature importance
rf = trained_models["Random Forest"]
importance = (
    pd.Series(rf.feature_importances_, index=feature_cols)
    .sort_values()
)
importance.plot(kind="barh", ax=axes[1])
axes[1].set_title("Random Forest Feature Importance")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("model_performance_results.png", dpi=250, bbox_inches="tight")
plt.show()

print("Top Random Forest features:")
display(importance.sort_values(ascending=False).to_frame("importance").head(9))


In [ ]:
# 12. Severe-day classification: Logistic Regression + SVM
# Project definition: severe day = Delhi PM2.5 > 150 µg/m³.
df_complete["is_severe"] = (df_complete["Delhi_PM2.5"] > 150).astype(int)

X_class = df_complete[feature_cols]
y_class = df_complete["is_severe"]

X_train_class = X_class.iloc[:split_idx]
X_test_class = X_class.iloc[split_idx:]
y_train_class = y_class.iloc[:split_idx]
y_test_class = y_class.iloc[split_idx:]

class_scaler = StandardScaler()
X_train_class_scaled = class_scaler.fit_transform(X_train_class)
X_test_class_scaled = class_scaler.transform(X_test_class)

weights = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_class),
    y=y_train_class
)
weight_dict = dict(zip(np.unique(y_train_class), weights))

logreg = LogisticRegression(
    random_state=42, max_iter=1000, class_weight=weight_dict, C=0.1
)
svm = SVC(
    kernel="rbf", C=1.0, class_weight="balanced",
    probability=True, random_state=42
)

classification_models = {
    "Logistic Regression": logreg,
    "SVM": svm
}

classification_results = []
for name, model in classification_models.items():
    model.fit(X_train_class_scaled, y_train_class)
    pred = model.predict(X_test_class_scaled)
    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_class, pred),
        "Precision": precision_score(y_test_class, pred),
        "Recall": recall_score(y_test_class, pred),
        "F1": f1_score(y_test_class, pred)
    })

classification_df = pd.DataFrame(classification_results)
print(
    f"Total days: {len(df_complete)} | "
    f"Severe: {y_class.sum()} ({y_class.mean()*100:.1f}%)"
)
display(classification_df.round(3))

joblib.dump(
    {
        "models": classification_models,
        "scaler": class_scaler,
        "feature_names": feature_cols
    },
    "classification_models.pkl"
)


In [ ]:
# 13. Final findings — one compact revision block
r1 = corr_df.query(
    "Era == 'Era 1 (2015–2019)' and Feature == 'Punjab_PM2.5_Lag_2Days'"
).iloc[0]
r2 = corr_df.query(
    "Era == 'Era 2 (2020–2023)' and Feature == 'Punjab_PM2.5_Lag_2Days'"
).iloc[0]

best = results_df.iloc[0]
rf_full = ablation_df.loc[ablation_df.Model == "Random Forest"].iloc[0]
lr_cls = classification_df.loc[classification_df.Model == "Logistic Regression"].iloc[0]
svm_cls = classification_df.loc[classification_df.Model == "SVM"].iloc[0]

print(f"""
🌫️ PROJECT TAKEAWAYS

1. Punjab → Delhi
   2-day lag correlation: {r1.r:.3f} → {r2.r:.3f}
   Change: {((r2.r-r1.r)/r1.r)*100:.1f}%

2. Haryana → Delhi
   1-day lag correlation in Era 2: {corr_df.query("Era == 'Era 2 (2020–2023)' and Feature == 'Haryana_PM2.5_Lag_1Day'").iloc[0].r:.3f}

3. Regression
   Best model: {best.Model}
   Test R²: {best.R2:.3f}
   Test RMSE: {best.RMSE:.2f}

4. Ablation
   Random Forest without Punjab: {rf_full.No_Punjab_R2:.3f}
   Relative reduction: {rf_full['Punjab_relative_drop_%']:.1f}%
   Random Forest without Haryana: {rf_full.No_Haryana_R2:.3f}
   Relative reduction: {rf_full['Haryana_relative_drop_%']:.1f}%

5. Severe-day classification
   Logistic Regression: accuracy {lr_cls.Accuracy:.3f}, recall {lr_cls.Recall:.3f}
   SVM: accuracy {svm_cls.Accuracy:.3f}, recall {svm_cls.Recall:.3f}

Interpretation:
• Punjab and Haryana contain useful predictive information.
• Haryana's lagged features are especially important in this model.
• Random Forest gives the strongest regression performance.
• SVM prioritizes recall, catching more severe days.
• These are associations/predictive relationships, not proof of causality.
""")

# Legacy comparison retained from the original notebook.
legacy = pd.DataFrame({
    "Analysis": ["OLD (80/20)", "NEW (70/30)"],
    "Best Model": ["Random Forest", "Random Forest"],
    "Test R2": [0.768, 0.814],
    "Test RMSE": [30.35, 29.08],
    "Punjab correlation": [0.612, 0.716],
    "Punjab contribution": ["Not calculated", "6.0%"],
    "Data quality": ["Basic", "Station-aware"]
})
display(legacy)

print("Dataset:", "https://www.kaggle.com/datasets/abhisheksjha/time-series-air-quality-data-of-india-2010-2023")
